In [1]:
%load_ext autoreload
%autoreload 2

## Imports

In [2]:
import os

from catboost import Pool, CatBoostClassifier
import numpy as np
import pandas as pd
from transformers import AutoTokenizer, AutoModel
import torch

## Read Data

In [4]:
train_data_bank = pd.read_csv("sentirueval/bank_train.csv", engine="c")
train_data_tkk = pd.read_csv("sentirueval/tkk_train.csv", engine="c")
train_data = pd.concat([train_data_bank, train_data_tkk], ignore_index=True)

test_data = pd.read_csv("sentirueval/bank_etalon.csv", engine="c")
print(f"Number of rows and columns in the train data set: {train_data.shape}")
print(f"Number of rows and columns in the test data set: {test_data.shape}")
print(train_data.head())
print(test_data.head())
# train_data.info()

Number of rows and columns in the train data set: (16743, 3)
Number of rows and columns in the test data set: (3212, 3)
   Unnamed: 0                                               text  label
0           0  http://t.co/YEVHuvVGA1 Взять кредит тюмень аль...      0
1           1  Мнение о кредитной карте втб 24 http://t.co/SB...      0
2           2  «Райффайзенбанк»: Снижение ключевой ставки ЦБ ...      0
3           3  Современное состояние кредитного поведения в р...      0
4           4  http://t.co/Qr6JbSVTxY Оформить краткосрочный ...      0
   Unnamed: 0                                               text  label
0           0           #Автокредит в россельхозбанк в череповце      0
1           1  RT @thomasabiloxuz: http://t.co/GTfMwSQQ2c #Кр...      0
2           2  #Автокредит в россельхозбанк 2012 http://t.co/...      0
3           3  RT @ronaldisogacoq: #Кредитные карты россельхо...      0
4           4  RT @anthonyogihulaf: #Кредиты в россельхозбанк...      0


## Preprocess Data

In [5]:
import re

def paragraph_clean(paragraph: str) -> list[str]:
    sentences = paragraph.strip().split(".")
    sentences = '. '.join(' '.join(i.strip().split()) for i in sentences if i)
    return sentences

def clean_text(text):
    text = re.sub(r'http\S+', '', text)  # Удаляем ссылки
    text = re.sub(r'[^\w\s]', '', text)  # Удаляем специальные символы
    text = text.lower()                  # Приводим к нижнему регистру
    return text


In [6]:
train_data_clean = train_data.copy()
# train_data_clean['text'] = train_data_clean['text'].apply(lambda x: clean_text(x))
train_data_list = [i for i in train_data_clean['text']]

test_data_clean = test_data.copy()
# test_data_clean['text'] = test_data_clean['text'].apply(lambda x: clean_text(x))
test_data_list = [i for i in test_data_clean['text']]


print(train_data_list[:3])
print(test_data_list[:3])

['http://t.co/YEVHuvVGA1 Взять кредит тюмень альфа банк', 'Мнение о кредитной карте втб 24 http://t.co/SBJTcsqjCg', '«Райффайзенбанк»: Снижение ключевой ставки ЦБ на заседании в эту пятницу очень маловероятно']
['#Автокредит в россельхозбанк в череповце', 'RT @thomasabiloxuz: http://t.co/GTfMwSQQ2c #Кредитный калькулятор россельхозбанк 2012', '#Автокредит в россельхозбанк 2012 http://t.co/9jc1OCzyv2']


## Getting Embeddings

In [13]:
model_path = 'ai-forever/FRIDA'
# model_path = 'sergeyzh/BERTA'

In [14]:
from sentence_transformers import SentenceTransformer

torch.set_float32_matmul_precision('highest')
device = "cuda:3" if torch.cuda.is_available() else "cpu"
print(device)
model = SentenceTransformer(model_path)
model = model.to(device)


cuda:3


README.md:   0%|          | 0.00/54.8k [00:00<?, ?B/s]

In [ ]:
model = model.to(device)
train_embeddings = model.encode(train_data_list)
test_embeddings = model.encode(test_data_list)
print(train_embeddings.shape)
print(test_embeddings.shape)
# with open('labse_embed.npy', 'wb+') as f:
#     np.save(f, train_embeddings)

## Catboost preprocess

In [12]:
# from catboost import Pool, CatBoostClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, classification_report
from skopt import BayesSearchCV
from skopt.space import Integer, Real, Categorical

IS_GRID_SEARCH = True

train_labels = train_data_clean["label"]
print(train_labels)
test_labels = test_data_clean["label"]
train_features = train_embeddings.copy()
test_features = test_embeddings.copy()
print(train_features)

cb_model = CatBoostClassifier(task_type="GPU",
                           devices='3',
                           auto_class_weights='Balanced',
                           random_seed=42,
                           verbose=False)
if IS_GRID_SEARCH:
    param_grid = {
                    'learning_rate': [0.1, 0.3],
                    'depth': [3, 5, 7], 
                    'l2_leaf_reg': [0.1, 1, 3], 
                    'min_data_in_leaf': [1, 3]  
                 }
    
    grid_search = GridSearchCV(
                                estimator=cb_model,
                                param_grid=param_grid,
                                scoring='f1_weighted',
                                cv=3, 
                                verbose=5,
                                n_jobs=1  # При использовании GPU лучше установить n_jobs=1
                               )
    
    grid_search.fit(train_features, train_labels)
    best_model = grid_search.best_estimator_

else:
    search_space = {
                'iterations': Integer(100, 1000),  # Количество итераций обучения
                'learning_rate': Real(0.01, 0.3, prior='log-uniform'),  # Скорость обучения
                'depth': Integer(4, 10),  # Глубина деревьев
                'l2_leaf_reg': Real(1e-8, 100, prior='log-uniform'),  # Регуляризация L2
                'random_strength': Real(1e-8, 10, prior='uniform'),  # Сила случайности при построении деревьев
                'grow_policy': Categorical(['SymmetricTree', 'Depthwise', 'Lossguide']),  # Политика роста деревьев
                'min_data_in_leaf': Integer(1, 10)  # Минимальное количество объектов в листе
            }


    bayes_search = BayesSearchCV(
        estimator=model,
        search_spaces=search_space,
        n_iter=50,  # Количество итераций байесовской оптимизации
        cv=4,  # Количество фолдов для кросс-валидации
        scoring='f1_micro',  # Метрика для оценки качества
        random_state=42,
        verbose=5,
        n_jobs=1  # Использование всех доступных ядер процессора
    )


    bayes_search.fit(train_features, train_labels)
    print("Наилучшие параметры:", bayes_search.best_params_)
    best_model = bayes_search.best_estimator_

y_pred = best_model.predict(test_features)
accuracy = classification_report(test_labels, y_pred)
print(accuracy)

0        0
1        0
2        0
3        0
4        0
        ..
16738    1
16739   -1
16740   -1
16741   -1
16742   -1
Name: label, Length: 16743, dtype: int64
[[-0.00490688  0.05256526  0.02427386 ... -0.03157939  0.04575169
   0.03150097]
 [ 0.01508501  0.03144686  0.0330537  ... -0.05179999  0.05060298
  -0.00842678]
 [-0.02609255  0.00974676  0.00381508 ...  0.0168619   0.03661052
   0.0192273 ]
 ...
 [ 0.03275232 -0.01338511  0.01239497 ...  0.01125143  0.01550651
  -0.00718946]
 [ 0.03062263 -0.00381071  0.00696217 ...  0.01420896  0.00789015
  -0.00243816]
 [ 0.02763271  0.01004685  0.01974866 ...  0.02081159  0.00073724
  -0.00916934]]
Fitting 3 folds for each of 36 candidates, totalling 108 fits


[CV 1/3] END depth=3, l2_leaf_reg=0.1, learning_rate=0.1, min_data_in_leaf=1;, score=0.717 total time=   4.6s


[CV 2/3] END depth=3, l2_leaf_reg=0.1, learning_rate=0.1, min_data_in_leaf=1;, score=0.771 total time=   4.8s


[CV 3/3] END depth=3, l2_leaf_reg=0.1, learning_rate=0.1, min_data_in_leaf=1;, score=0.609 total time=   4.9s


[CV 1/3] END depth=3, l2_leaf_reg=0.1, learning_rate=0.1, min_data_in_leaf=3;, score=0.717 total time=   5.0s


...

[CV 2/3] END depth=3, l2_leaf_reg=1, learning_rate=0.1, min_data_in_leaf=1;, score=0.770 total time=   4.7s


[CV 3/3] END depth=3, l2_leaf_reg=1, learning_rate=0.1, min_data_in_leaf=1;, score=0.609 total time=   4.5s


[CV 1/3] END depth=3, l2_leaf_reg=1, learning_rate=0.1, min_data_in_leaf=3;, score=0.715 total time=   4.4s


[CV 2/3] END depth=3, l2_leaf_reg=1, learning_rate=0.1, min_data_in_leaf=3;, score=0.770 total time=   4.3s
[CV 3/3] END depth=3, l2_leaf_reg=1, learning_rate=0.1, min_data_in_leaf=3;, score=0.609 total time=   4.4s
[CV 1/3] END depth=3, l2_leaf_reg=1, learning_rate=0.3, min_data_in_leaf=1;, score=0.698 total time=   4.3s
[CV 2/3] END depth=3, l2_leaf_reg=1, learning_rate=0.3, min_data_in_leaf=1;, score=0.772 total time=   4.1s
[CV 3/3] END depth=3, l2_leaf_reg=1, learning_rate=0.3, min_data_in_leaf=1;, score=0.671 total time=   4.2s
[CV 1/3] END depth=3, l2_leaf_reg=1, learning_rate=0.3, min_data_in_leaf=3;, score=0.698 total time=   4.2s
[CV 2/3] END depth=3, l2_leaf_reg=1, learning_rate=0.3, min_data_in_leaf=3;, score=0.772 total time=   4.2s
[CV 3/3] END depth=3, l2_leaf_reg=1, learning_rate=0.3, min_data_in_leaf=3;, score=0.671 total time=   4.3s
[CV 1/3] END depth=3, l2_leaf_reg=3, learning_rate=0.1, min_data_in_leaf=1;, score=0.710 total time=   4.2s
[CV 2/3] END depth=3, l2_lea

KeyboardInterrupt: 

In [ ]:
print(grid_search.best_params_)

In [ ]:
from sklearn.metrics import f1_score, precision_score, recall_score
f1_macro = f1_score(test_labels, y_pred, average='macro')
f1_micro = f1_score(test_labels, y_pred, average='micro')
f1_weighted = f1_score(test_labels, y_pred, average='weighted')
print(f"{f1_macro=}, {f1_micro=}, {f1_weighted=}")
precision = precision_score(test_labels, y_pred, average='macro')
recall = recall_score(test_labels, y_pred, average='macro')
print(f"{precision=}, {recall=}, {f1_macro=}, {f1_micro=}, {f1_weighted=}")

In [ ]:
cb_model = CatBoostClassifier(task_type="GPU",
                           devices='1',
                           random_seed=42,
                           verbose=True,
                            depth=3, l2_leaf_reg=1, learning_rate=0.1, min_data_in_leaf=3)
cb_model.fit(X_train,
          y_train)

y_pred = cb_model.predict(X_val)
accuracy = classification_report(y_val, y_pred)
print(accuracy)
